In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import emcee
import corner
import os
from astropy.cosmology import FlatLambdaCDM, FlatwCDM
import astropy.units as u

os.makedirs('plots',  exist_ok=True)
os.makedirs('chains', exist_ok=True)

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

In [2]:
# Read the data
df   = pd.read_csv('data/Pantheon+SH0ES.dat', sep='\s+', comment='#')
mask = (df['zHD'] > 0.01) & (df['IS_CALIBRATOR'] == 0)
df   = df[mask].reset_index(drop=True)

z_sn   = df['zHD'].values
mB     = df['mB'].values
x1     = df['x1'].values
c_     = df['c'].values
N_SNe  = len(z_sn)

with open('data/Pantheon+SH0ES_STATONLY.cov') as f:
    N_cov  = int(f.readline())
    C_full = np.array(f.read().split(), dtype=float).reshape(N_cov, N_cov)
orig_mask = (pd.read_csv('data/Pantheon+SH0ES.dat', sep='\s+', comment='#')
             .pipe(lambda d: (d['zHD']>0.01)&(d['IS_CALIBRATOR']==0)).values)
idx       = np.where(orig_mask)[0]
C_sn      = C_full[np.ix_(idx, idx)]
C_sn_inv  = np.linalg.inv(C_sn)
_, logdet_sn = np.linalg.slogdet(2*np.pi*C_sn)

M_fixed = -19.253   # Riess et al. 2022 / SH0ES (arXiv:2112.04510)

print(f'SNe Ia in fit: {N_SNe}   z range: {z_sn.min():.4f}-{z_sn.max():.4f}')

<>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:15: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:15: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
/var/folders/sc/3sbv5zgs77zcpmwsqg_yfb500000gn/T/ipykernel_68150/538109132.py:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  df   = pd.read_csv('data/Pantheon+SH0ES.dat', sep='\s+', comment='#')
/var/folders/sc/3sbv5zgs77zcpmwsqg_yfb500000gn/T/ipykernel_68150/538109132.py:15: Synt

SNe Ia in fit: 1580   z range: 0.0102-2.2614


In [3]:
# Table I of Chen, Huang & Wang (2019, arXiv:1808.05724), Planck 2018 TT,TE,EE+lowE
R_PLANCK, R_ERR       = 1.7502, 0.0046
OBH2_PLANCK, OBH2_ERR = 0.02236, 0.00015
RHO_R_OBH2            = -0.66

d_cmb   = np.array([R_PLANCK, OBH2_PLANCK])
cov_cmb = np.array([
    [R_ERR**2,                       RHO_R_OBH2*R_ERR*OBH2_ERR],
    [RHO_R_OBH2*R_ERR*OBH2_ERR,       OBH2_ERR**2]
])
cov_cmb_inv = np.linalg.inv(cov_cmb)
_, logdet_cmb = np.linalg.slogdet(2*np.pi*cov_cmb)

C_LIGHT = 299792.458   # km/s

def z_star_fit(Obh2, Omh2):
    """Recombination redshift fitting formula (Hu & Sugiyama 1996)."""
    g1 = 0.0783*(Obh2)**(-0.238) / (1 + 39.5*(Obh2)**0.763)
    g2 = 0.560 / (1 + 21.1*(Obh2)**1.81)
    return 1048*(1+0.00124*(Obh2)**(-0.738))*(1+g1*(Omh2)**g2)

def shift_parameter_R(H0, Om0, Obh2):
    """CMB shift parameter R(z*) for flat cosmologies. Uses astropy for D_A."""
    h     = H0/100
    Omh2  = Om0 * h**2
    zstar = z_star_fit(Obh2, Omh2)
    cosmo = FlatLambdaCDM(H0=H0, Om0=Om0)
    DA    = cosmo.angular_diameter_distance(zstar).to(u.Mpc).value
    return (1+zstar) * DA * np.sqrt(Om0) * H0 / C_LIGHT

def log_likelihood_cmb(H0, Om0, Obh2):
    R    = shift_parameter_R(H0, Om0, Obh2)
    x    = np.array([R, Obh2])
    delta = x - d_cmb
    return -0.5 * (delta @ cov_cmb_inv @ delta + logdet_cmb)

In [4]:
# Validarion of shift parameter calculation against Planck 2018 results
H0_fid, Om0_fid, Obh2_fid = 67.36, 0.3153, 0.02237
R_check     = shift_parameter_R(H0_fid, Om0_fid, Obh2_fid)
zstar_check = z_star_fit(Obh2_fid, Om0_fid*(H0_fid/100)**2)

print(f'z_star computed : {zstar_check:.2f}   (Planck reference: ~1089.9)')
print(f'R computed      : {R_check:.4f}   (Chen+2019 reference: {R_PLANCK})')
print(f'Relative error  : {100*abs(R_check-R_PLANCK)/R_PLANCK:.2f}%')

z_star computed : 1091.91   (Planck reference: ~1089.9)
R computed      : 1.7582   (Chen+2019 reference: 1.7502)
Relative error  : 0.46%


In [5]:
# Load priors and define likelihoods for SNe
BOUNDS = {
    'H0':    (50, 100),
    'Om0':   (0.05, 0.7),
    'w':     (-3.0, 0.0),
    'Obh2':  (0.018, 0.026),
    'alpha': (0.0, 1.0),
    'beta':  (1.0, 5.0),
}

def mu_th(model, H0, Om0, w=None):
    if model == 'LCDM':
        cosmo = FlatLambdaCDM(H0=H0, Om0=Om0)
    else:
        cosmo = FlatwCDM(H0=H0, Om0=Om0, w0=w)
    return 5*np.log10(cosmo.luminosity_distance(z_sn).to(u.Mpc).value) + 25

def log_likelihood_sne(mu_theory, alpha, beta):
    mu_o  = mB + alpha*x1 - beta*c_ - M_fixed
    delta = mu_o - mu_theory
    return -0.5 * (delta @ C_sn_inv @ delta + logdet_sn)

def make_log_prob(model, use_cmb):
    """
    Builds a log-probability function for a given model ('LCDM' or 'wCDM')
    and whether to include the CMB distance prior.
    Parameter order: LCDM -> [H0, Om0, alpha, beta]      (+Obh2 if use_cmb)
                      wCDM -> [H0, Om0, w, alpha, beta]  (+Obh2 if use_cmb)
    """
    def log_prob(theta):
        if model == 'LCDM':
            if use_cmb:
                H0, Om0, alpha, beta, Obh2 = theta
            else:
                H0, Om0, alpha, beta = theta
            w = None
        else:
            if use_cmb:
                H0, Om0, w, alpha, beta, Obh2 = theta
            else:
                H0, Om0, w, alpha, beta = theta

        if not (BOUNDS['H0'][0]<H0<BOUNDS['H0'][1] and BOUNDS['Om0'][0]<Om0<BOUNDS['Om0'][1]
                and BOUNDS['alpha'][0]<alpha<BOUNDS['alpha'][1] and BOUNDS['beta'][0]<beta<BOUNDS['beta'][1]):
            return -np.inf
        if model == 'wCDM' and not (BOUNDS['w'][0]<w<BOUNDS['w'][1]):
            return -np.inf
        if use_cmb and not (BOUNDS['Obh2'][0]<Obh2<BOUNDS['Obh2'][1]):
            return -np.inf

        lp = log_likelihood_sne(mu_th(model, H0, Om0, w), alpha, beta)
        if use_cmb:
            lp += log_likelihood_cmb(H0, Om0, Obh2)
        return lp
    return log_prob

In [6]:
# emcee sampler setup
def run_emcee(log_prob, theta0, scatter, nwalkers=64, nsteps=6000, warmup=1500):
    ndim = len(theta0)
    rng  = np.random.default_rng(42)
    pos  = np.empty((nwalkers, ndim))
    n = 0
    for _ in range(200_000):
        if n == nwalkers: break
        cand = theta0 + scatter * rng.standard_normal(ndim)
        if np.isfinite(log_prob(cand)):
            pos[n] = cand; n += 1
    if n < nwalkers:
        raise RuntimeError(f'Only {n}/{nwalkers} valid starting positions found')

    moves   = [(emcee.moves.DEMove(), 0.8), (emcee.moves.DESnookerMove(), 0.2)]
    sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob, moves=moves)
    state   = sampler.run_mcmc(pos, warmup, progress=False)
    sampler.reset()
    sampler.run_mcmc(state, nsteps, progress=True)
    return sampler

def get_flat(sampler, burnin_factor=3):
    try:
        tau    = np.nan_to_num(sampler.get_autocorr_time(quiet=True), nan=100.0)
        burnin = int(burnin_factor*tau.max()); thin = max(1, int(tau.max()/2))
    except Exception:
        burnin, thin = 500, 10
    return sampler.get_chain(discard=burnin, thin=thin, flat=True), burnin, thin

def convergence_report(sampler, flat, label):
    chain = sampler.get_chain()
    af    = sampler.acceptance_fraction
    print(f'\n-- {label} --')
    print(f'  Acceptance fraction : mean={af.mean():.3f}')
    try:
        tau = np.nan_to_num(sampler.get_autocorr_time(quiet=True), nan=np.nan)
        print(f'  Autocorr time tau   : {np.round(tau,1)}')
        print(f'  Steps/tau (>50 ok)  : {np.round(chain.shape[0]/tau,1)}')
        print(f'  Effective samples   : {np.round(flat.shape[0]/tau).astype(int)}')
    except Exception:
        print('  Autocorr time       : could not compute')
    N, M_w, D = chain.shape
    W    = chain.var(axis=0, ddof=1).mean(axis=0)
    B    = N*chain.mean(axis=0).var(axis=0, ddof=1)
    Rhat = np.sqrt((N-1)/N + B/(N*W))
    print(f'  Gelman-Rubin R-hat  : {np.round(Rhat,4)}  (<1.01 converged)')

def print_results(flat, labels):
    print(f'  {"Param":>10}  {"16%":>9}  {"Median":>9}  {"84%":>9}')
    for i, lbl in enumerate(labels):
        p16,p50,p84 = np.percentile(flat[:,i], [16,50,84])
        print(f'  {lbl:>10}  {p16:>9.4f}  {p50:>9.4f}  {p84:>9.4f}')